# DS2002 · Reflection What Broke What Mattered

**Studio — 2026-11-04 · Fall 2026**  
**Class time:** 45 minutes

---

## Reflection — what broke, what mattered

The project is submitted. This hour is for extracting what is reusable from it, because the capstone starts in under two weeks and it is the same work at larger scale.

Write individually for the first fifteen minutes, then compare with your team. The disagreements are the interesting part — if two people on the same team have different accounts of which decision mattered most, at least one of you learned something the other did not.

Answers here should be specific enough that someone could act on them. "Communicate better" is not usable. "Name one notebook owner per week, because we spent two hours reconciling two versions of the cleaning" is.

### 1. Data quality

Which single data-quality problem, if it had gone unnoticed, would have changed your conclusion the most? Give the number both ways.

**SKU fragmentation: Strawberry Pop-Tarts split across 4 SKUs (`PT-12`, `SKU#459812`, `POPTART-STRAW`, `PT_STRAWBERRY`) and 6 name spellings.** Left unnoticed, it flips the main storm-week finding:

- **With the fix:** Pop-Tarts were the **#1 product in the 7 days before landfall (Sept 19–25), at 377 units/day**, ahead of bottled water at 320/day. Over the month they were #2 of 20 products (5,198 units).
- **Without it:** the best single Pop-Tart SKU is **#3 at 103 units/day**, behind two bottled-water SKUs. Over the month it is #8 of 45 SKUs (1,381 units), so water looks like the whole story.

Fragmentation did *not* hide the surge rate. Each fragment still shows about 5× its baseline. What it hid was the size of the demand, and the size is what a stocking decision runs on.

(For the checkpoint below I used the category-label problem from question 2, because it is the one that changed the numbers in my own analysis. SKU fragmentation is the one that would have changed the team's conclusion most.)

### 2. ETL trade-offs

Name a cleaning decision where a reasonable person could have chosen the other way. What was the other option, what would it have done to your reported numbers, and would you make the same call again?

**Merging "Games & Entertainment" into Entertainment and "Health & Wellness" into Health.** Our `normalize_category` function missed these two labels, so the main cleaning kept them as separate categories, and they were only folded in later, inside the Q3 analysis (472 rows).

Keeping source labels as recorded is a defensible choice: you shouldn't merge categories without evidence that they mean the same thing. The other way changes Q1, though (surge multiple, Sept 19–25 vs Sept 1–15):

| | Unmerged | Merged |
|---|---|---|
| Entertainment | 2.13×, plus "Games & Entertainment" at **2.63×** | **2.25×** |
| Health | 2.17×, plus "Health & Wellness" at **2.51×** | **2.24×** |
| Q1 ranking | the two small labels rank **#3 and #4**, ahead of Electronics (2.50×) | Electronics stays #3 |

The small labels' higher multiples come from a small baseline: 24 and 9 units/day, against 78 and 33 for the main labels. What settled it was checking the products under each label. "Games & Entertainment" holds only Monopoly and playing cards, exactly like "Entertainment", and "Health & Wellness" holds only the first-aid kit, like "Health". They are spelling variants, not similar categories.

**Same call again?** Yes, but in the cleaning step, with an assertion on the exact category list, not patched in during analysis. Applied there, Q1 and Q3 would have used the same categories from the start.

### 3. Pipeline trust

If someone handed you this notebook and asked whether the numbers are right, what would you check first? Is that check currently in the notebook? If not, that is your first capstone improvement.

**First check: is this the right version of the data, and do the row counts survive every step?**

The second half is in the team notebook. It asserts unique `transaction_id`s, unit totals through the daily rollups, 360 → 360 store-days and unchanged units through the weather join, 24 hours per store-day, and that the API's locations come back in the requested order.

The first half is **not** in the team notebook. The course repo's `data/` folder and our team repo have the same 25,562 transactions with every timestamp **11 days apart** (sales peak Sept 13–14 vs Sept 24–25). The instructor re-timed the data to line up with Helene's Sept 26 landfall, and only our team repo has the new version. The old copy loads, cleans and joins without a single error, but it puts the buying surge two weeks *before* the storm, so the Q3 "buying leads the weather by 2 days" result would come out as something meaningless.

I added a guard to my copy of the progress lab: it loads the data from the team repo pinned to commit `fd1583a` and asserts the sales peak falls on Sept 25. **First capstone improvement:** a data fingerprint cell at the top of the main notebook. It would check the pinned source, the row count, the date range, the expected peak date and the expected category list, so loading the wrong data fails loudly instead of quietly producing different numbers.

### 4. Business versus data

Your recommendation went past what the data strictly proves — every useful recommendation does. Where exactly is that line in your work, and how would you say it out loud to someone who is about to spend money based on it?

**What the data shows:** in one storm (Helene, Sept 2024) across 12 Florida stores:
- Pop-Tarts sold 5.1× their Sept 1–15 baseline in the 7 days before landfall.
- Every store's sales peaked 1–3 days before its own worst weather.
- Wind correlates with sales at r = +0.80 when the weather is measured 2 days later (n = 28 days).

**Where the line is:** "stock Pop-Tarts at about 5× and have them on shelves before the wind arrives" turns one storm into a rule. Three things say to hold it loosely:
1. **It is one event.** The rank (Spearman) correlation at the same 2-day lag is only +0.25, so the 0.80 comes from Helene's few extreme days, not from a steady weather-to-sales relationship.
2. **The multiple depends on how we define it.** It is 5.1× with a Sept 1–15 baseline, 4.3× with Sept 1–18, and 7.7× if you take the single peak day.
3. **There is a step we can't explain.** Sales jumped about 1.4× on Sept 16 at all 12 stores, ten days before landfall and before Helene was named.

**Said out loud:** "In Helene, Pop-Tarts ran about five times normal the week before landfall, and buying peaked about two days before the worst weather. That's one storm, not a law. I'd plan for four to five times normal and have it in stores three days before landfall, then use the first two days of actual sales to decide on the rest rather than committing the whole order up front."

### 5. Team process

What cost your team the most time that was not a data problem? Merge conflicts, duplicated work, waiting on someone, unclear ownership? Name the specific change you will make for the capstone.

**Repo organization with four people working in it.** We never agreed on where things go, so the repo collected several kinds of mess:

- **Duplicate notebooks.** The same progress lab existed under two filenames (underscores vs em dashes) in two folders (`checkpoints/` and `notebooks/06-midterm-walmart/`), and a duplicate checkpoint `(1).ipynb` had to be deleted.
- **Junk committed by accident.** A copy of the repo itself got committed inside `notebooks/06-midterm-walmart/`, along with a `.DS_Store` file.
- **Analysis spread across files.** Q3 moved out of `midterm_notebook.ipynb` and into the lab notebook during cleanup, so the file the README calls "our final notebook" no longer holds every answer.
- **Copy-pasted cleaning code.** It was copied into each lab, which is how the category-label fix ended up living only inside Q3.

A lot of the time went into working out *which* file was current, not into the data.

**Change for the capstone:** on day one, write the layout into the README and hold to it:
- `data/` holds one pinned version of the data.
- `src/cleaning.py` holds the shared cleaning and API code that every notebook imports.
- One `notebooks/main.ipynb` is the deliverable.
- `checkpoints/` holds class submissions, under the course's exact filenames.
- A `.gitignore` covers `.DS_Store` and nested repos.

Each person works in their own branch or notebook, and one owner per week merges into `main.ipynb`.

### 6. Carry-forward list

**TODO:** three concrete things you are taking into the capstone. Code you will reuse, a habit you are adopting, or a mistake you are not repeating.

Run the cell — this list is your capstone kickoff agenda, so make it real.

In [1]:
carry_forward = [
    'Reuse: our cleaning functions (normalize_missing, normalize_store_id, normalize_category, to_number) and the '
    'retrying Open-Meteo fetch with the UTC-to-Eastern conversion, moved into one shared cleaning.py that every '
    'notebook imports instead of copy-pasting it into each lab',
    'Habit: a data fingerprint at the top (pinned source, row count, date range, expected categories) and an '
    'assertion after every load, join and rollup that row counts and unit totals did not change',
    'Mistake not repeating: defining the headline metric differently in different places. We reported 7.7x '
    '(peak day, Sept 1-18 baseline) at the 10/28 checkpoint and 5.1x (7-day average, Sept 1-15 baseline) in the '
    'progress lab. Next time the windows and baseline go in the README before anyone computes a number',
]

for i, item in enumerate(carry_forward, 1):
    flag = 'unfilled' if item.startswith('TODO') else 'ok'
    print(f'{i}. [{flag}] {item}')
print()
print('bring this list to the capstone kickoff on Nov 16.')

1. [ok] Reuse: our cleaning functions (normalize_missing, normalize_store_id, normalize_category, to_number) and the retrying Open-Meteo fetch with the UTC-to-Eastern conversion, moved into one shared cleaning.py that every notebook imports instead of copy-pasting it into each lab
2. [ok] Habit: a data fingerprint at the top (pinned source, row count, date range, expected categories) and an assertion after every load, join and rollup that row counts and unit totals did not change
3. [ok] Mistake not repeating: defining the headline metric differently in different places. We reported 7.7x (peak day, Sept 1-18 baseline) at the 10/28 checkpoint and 5.1x (7-day average, Sept 1-15 baseline) in the progress lab. Next time the windows and baseline go in the README before anyone computes a number

bring this list to the capstone kickoff on Nov 16.


---

## Checkpoint (participation)

Report the single data-quality problem that mattered most and the number it would have changed.

Work the last few minutes in groups of 4–5, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Wednesday 11:59pm ET**. One submission per person, not per group.

In [2]:
# Checkpoint
problem_that_mattered = ('Unmerged category labels: "Games & Entertainment" and "Health & Wellness" left separate '
                         'from Entertainment and Health (same products: Monopoly/cards, first-aid kit)')
number_with_fix = 'Entertainment 2.25x, Health 2.24x surge (Sep 19-25 vs Sep 1-15); Electronics is #3 at 2.50x'
number_without_fix = '"Games & Entertainment" 2.63x and "Health & Wellness" 2.51x, ranked #3 and #4 ahead of Electronics'
capstone_change = ('One owner for the shared cleaning code: any label fix found during analysis goes back into the '
                   'cleaning step, with an assertion on the exact category list, instead of being patched inside one question')

print('problem:', problem_that_mattered)
print('reported:', number_with_fix, 'vs unfixed:', number_without_fix)
print('changing for the capstone:', capstone_change)

problem: Unmerged category labels: "Games & Entertainment" and "Health & Wellness" left separate from Entertainment and Health (same products: Monopoly/cards, first-aid kit)
reported: Entertainment 2.25x, Health 2.24x surge (Sep 19-25 vs Sep 1-15); Electronics is #3 at 2.50x vs unfixed: "Games & Entertainment" 2.63x and "Health & Wellness" 2.51x, ranked #3 and #4 ahead of Electronics
changing for the capstone: One owner for the shared cleaning code: any label fix found during analysis goes back into the cleaning step, with an assertion on the exact category list, instead of being patched inside one question
